# 03 贪吃蛇GUI游戏开发


## 1. 案例目标与资料范围

本篇对应《VibeCoding实战》的视频 **07—入门案例：贪吃蛇GUI游戏开发**。在完成上一篇的环境准备后，用Claude Code开发一个可以通过键盘控制的桌面游戏。

**GUI**是图形用户界面。本案例使用 **Python + Pygame**，Pygame负责窗口、图形、键盘事件和计时。程序运行在本地，不需要浏览器，也不需要在游戏运行时调用大模型。

本篇已读取网盘中`项目/snake/snake_game.py`的完整代码（420行）和`requirements.txt`。下面按“准备项目 → 启动游戏 → 理解控制流程 → 核对行为”展开，代码片段来自该项目。

> 文件开头写着“撞到墙壁或自身则游戏结束”，实际`Snake.move()`采用取模实现穿墙。本篇按源码实际行为讲解，并保留这个说明与实现不一致的地方。

配套语雀的「入门实战项目」是个人记账APP，对应后续笔记；本篇不混入它的技术方案。本次没有逐段观看视频，因此不把补充提示词或运行步骤写成课堂逐字记录。


## 2. 准备项目与依赖

网盘提供`项目/snake`文件夹和`snake.rar`。将项目放到本地独立目录，并用VSCode打开。Windows以下以`C:\dev\snake`为例，macOS以`~/dev/snake`为例；路径按自己的存放位置替换。

源项目的文件结构：

```text
snake/
├── snake_game.py    游戏入口和全部游戏代码
├── requirements.txt 依赖记录
├── high_score.txt   最高分记录
├── .claude/         Claude Code项目配置目录
└── __pycache__/     Python生成的缓存
```

`requirements.txt`原文：

```text
pygame>=2.5.0
```

`>=`表示最低版本要求，不是固定安装2.5.0。`random`、`enum`和`collections`属于Python标准库，无需另外安装。原项目没有`pyproject.toml`，本篇保留它的依赖文件，按本仓库习惯用uv创建环境和安装依赖。

### Windows PowerShell

先进入包含`snake_game.py`和`requirements.txt`的目录：

```powershell
Set-Location -LiteralPath 'C:\dev\snake'
Get-ChildItem
uv --version
uv venv .venv
uv pip install --python .\.venv\Scripts\python.exe -r requirements.txt
.\.venv\Scripts\python.exe -c "import pygame; print(pygame.version.ver)"
```

### macOS

```bash
cd ~/dev/snake
ls
uv --version
uv venv .venv
uv pip install --python ./.venv/bin/python -r requirements.txt
./.venv/bin/python -c "import pygame; print(pygame.version.ver)"
```

以上假设uv已经可用。若已创建本项目的`.venv`，检查并使用它即可。安装失败时先看所选Python版本、Pygame版本和平台是否匹配，不必重装全部开发工具。

最后一条命令用于确认这个解释器能导入Pygame；它还不能证明键盘和游戏界面正常。


## 3. 向Claude Code说明任务

本案例的开发结果是一个小型桌面游戏。描述任务时需要明确操作键、吃食物后的变化、边界规则和结束后的行为，避免仅说“做个贪吃蛇”让AI自行猜测。

**以下是依据现有源码整理的复现提示词，属于笔记补充，不是视频原提示词。**

```text
在当前目录使用Python和Pygame开发一个贪吃蛇桌面游戏。

要求：
1. 使用30列、20行的网格，每格20像素。
2. 初始蛇长为3，向右移动；方向键和WASD控制方向，禁止直接反向。
3. 到达边界后从另一侧进入，碰到自身则游戏结束。
4. 吃食物加10分并增长，食物生成在未被蛇占据的格子。
5. P暂停或继续，R重新开始，ESC或关闭窗口退出。
6. 初始移动间隔150毫秒，每增加50分缩短10毫秒，最短50毫秒；
   按住空格临时加速。
7. 显示分数、最高分、速度和暂停或结束提示；
   最高分使用high_score.txt保存。

先说明方案，再生成代码与requirements.txt。
完成后说明运行方式、实际检查结果，以及需要人工操作验证的项目。
```

如果已经打开的是课程源码，应让AI先阅读并解释现有文件，再处理指定问题。不要把“重新生成一份”与“复现原课程代码”混为同一个任务。


## 4. 启动游戏与键盘操作

在项目根目录运行入口文件。Windows：

```powershell
.\.venv\Scripts\python.exe snake_game.py
```

macOS：

```bash
./.venv/bin/python snake_game.py
```

程序应创建一个 **600×400像素**的窗口。蛇初始长度为3，头部位于网格`(15, 10)`，初始向右移动；不是等待点击开始按钮的版本。

| 输入 | 实际作用 |
|---|---|
| ↑↓←→或WASD | 改变下一步方向 |
| P | 暂停或继续，游戏结束后不处理此操作 |
| R | 重置并立即开始新一局 |
| 按住空格 | 时间累计倍率变为2，临时加快移动 |
| ESC | 退出 |
| 关闭窗口 | 退出 |

开始操作前先让游戏窗口获得键盘焦点。结束后回到终端查看是否有异常信息。

运行时保持当前目录为项目目录，因为源码直接用相对路径`high_score.txt`读写最高分；从其他目录执行脚本会改变该文件的查找和保存位置。


## 5. 从入口看整体执行过程

代码把职责分成四个类型：

| 类型 | 主要职责 |
|---|---|
| `Direction` | 定义上、下、左、右对应的位移 |
| `Snake` | 管理蛇身、方向、移动、增长和自身碰撞 |
| `Food` | 在空格生成食物，绘制食物及动画 |
| `Game` | 创建窗口，处理输入、计时、计分、状态和绘制 |

**类**把一组状态和相关操作放在一起；创建实例后，不同对象各自持有状态。本例的`Game`持有一个`Snake`和一个`Food`。

入口代码：

```python
if __name__ == "__main__":
    game = Game()
    game.run()
```

直接运行文件时创建`Game`，随后进入主循环。导入模块时，这段入口不会自动执行。

`Game.run()`原代码：

```python
def run(self):
    """主游戏循环"""
    while self.running:
        dt = self.clock.tick(FPS)
        self.handle_events()
        self.update(dt)
        self.draw()

    self._save_high_score()
    pygame.quit()
```

每轮按顺序处理输入、更新逻辑、绘制画面。`dt`是这一轮与上一轮之间经过的毫秒数；`clock.tick(FPS)`同时限制循环帧率。`FPS=60`指每秒最多约60轮画面循环，不表示蛇每秒移动60格。

`running=False`让循环结束，随后保存最高分并释放Pygame资源。


## 6. 网格坐标与移动方向

游戏规则使用“第几格”，绘制画面使用“第几个像素”。两者通过每格大小转换。

源码常量：

```python
CELL_SIZE = 20          # 每个格子的大小（像素）
GRID_WIDTH = 30         # 网格宽度（格子数）
GRID_HEIGHT = 20        # 网格高度（格子数）
WINDOW_WIDTH = CELL_SIZE * GRID_WIDTH
WINDOW_HEIGHT = CELL_SIZE * GRID_HEIGHT

FPS = 60                # 帧率
```

| 项目 | 范围或换算 |
|---|---|
| 网格横坐标x | 0—29 |
| 网格纵坐标y | 0—19 |
| 屏幕坐标原点 | 左上角 |
| 向右、向下 | x增加、y增加 |
| 网格到像素 | `(x * CELL_SIZE, y * CELL_SIZE)` |

例如网格`(3, 2)`对应的矩形左上角是像素`(60, 40)`，矩形大小为20×20。

方向定义：

```python
class Direction(Enum):
    """移动方向枚举"""
    UP = (0, -1)
    DOWN = (0, 1)
    LEFT = (-1, 0)
    RIGHT = (1, 0)
```

`Enum`是枚举，用名字表示有限的取值。向右移动一步，就是给头部坐标加`(1, 0)`。


## 7. 蛇身存储与禁止直接反向

蛇身是从蛇头到蛇尾的一组格子坐标。源码使用`deque`（双端队列），便于在头部插入坐标、在尾部删除坐标。

`Snake.__init__()`和`head`属性：

```python
def __init__(self):
    # 蛇身：deque 方便在头部添加、尾部删除
    # 初始位置：屏幕中央偏左，长度为3
    start_x = GRID_WIDTH // 2
    start_y = GRID_HEIGHT // 2
    self.body = deque([
        (start_x, start_y),
        (start_x - 1, start_y),
        (start_x - 2, start_y),
    ])
    self.direction = Direction.RIGHT
    self.next_direction = Direction.RIGHT  # 缓冲输入，防止瞬间反向
    self.growing = False

@property
def head(self):
    return self.body[0]
```

初始蛇身为`[(15, 10), (14, 10), (13, 10)]`。`direction`记录当前已采用的方向，`next_direction`缓存下一次移动准备采用的方向；`growing`记录下一步是否保留尾巴。

`set_direction()`用当前方向阻止180°转向：

```python
def set_direction(self, new_dir: Direction):
    """设置下一步方向（阻止直接反向）"""
    # 禁止 180° 转向
    opposites = {
        Direction.UP: Direction.DOWN,
        Direction.DOWN: Direction.UP,
        Direction.LEFT: Direction.RIGHT,
        Direction.RIGHT: Direction.LEFT,
    }
    if new_dir != opposites.get(self.direction):
        self.next_direction = new_dir
```

正在向右走时，按左不会生效，按上或下才会改变待执行方向。判断基于`direction`，真正移动时才把`next_direction`应用到当前方向。


## 8. 移动、穿墙与自身碰撞

普通移动时，加上新蛇头，再删除原蛇尾，长度不变；增长时只加头、不删尾。

`Snake.move()`原代码：

```python
def move(self):
    """移动蛇，返回是否存活"""
    self.direction = self.next_direction
    dx, dy = self.direction.value
    hx, hy = self.head
    new_head = (hx + dx, hy + dy)

    # 穿墙：从一侧出去，从另一侧进来
    new_head = (new_head[0] % GRID_WIDTH, new_head[1] % GRID_HEIGHT)

    # 撞自身检测（检查是否碰到除尾部以外的身体）
    # 如果正在生长，尾巴不会缩，所以要检查完整身体
    if self.growing:
        if new_head in self.body:
            return False
    else:
        if new_head in list(self.body)[:-1]:
            return False

    self.body.appendleft(new_head)

    if self.growing:
        self.growing = False
    else:
        self.body.pop()

    return True
```

### 穿墙

`%`是取模。宽度30时，`30 % 30 = 0`，`-1 % 30 = 29`，所以从右边界出去会回到最左侧，从左边界出去会回到最右侧；纵坐标同理。

### 为什么普通移动不检查原蛇尾

如果本步不增长，原尾部会在本步离开，所以头部进入那个格子是允许的；增长时尾部仍占着格子，需要检查完整身体。`list(self.body)[:-1]`表示除去最后一个尾部坐标。

碰撞时返回`False`，不会写入新蛇头。`Game.update()`根据返回值设置`game_over=True`，而不是立即关闭窗口。

`grow()`只是设置标记：

```python
def grow(self):
    """标记下一次移动时生长"""
    self.growing = True
```

源码先移动，再判断是否吃到食物；因此吃到时马上加分，长度在**下一次成功移动**时增加。这是当前实现的具体时序。


## 9. 在空格生成食物

如果食物生成在蛇身上，玩家就无法正常获取它。因此源码先枚举所有格子，再去掉蛇占据的位置。

`Food.spawn()`：

```python
def spawn(self, snake_body, current_time):
    """在未被蛇占据的位置生成食物"""
    all_positions = {(x, y) for x in range(GRID_WIDTH) for y in range(GRID_HEIGHT)}
    available = all_positions - set(snake_body)
    if available:
        self.position = random.choice(list(available))
        self.spawn_time = current_time
```

| 表达式 | 含义 |
|---|---|
| `all_positions` | 30×20，共600个合法格子 |
| `set(snake_body)` | 蛇占据的坐标集合 |
| `all_positions - set(snake_body)` | 未被占据的格子 |
| `random.choice(...)` | 从候选格子中随机取一个 |

**集合**适合表示不重复的坐标和计算差集。逻辑碰撞判断使用`Food.position`，绘制时再转换成像素位置。

当前`if available`没有对应的`else`。棋盘占满时，不会生成新位置，也没有单独的胜利处理；阅读源码时应把这一边界与正常生成流程分开。


## 10. 键盘事件、暂停与重开

`Game.handle_events()`先读取事件队列，再检查持续按住的按键。**事件队列**保存窗口关闭、按键按下等已发生的输入；持续按键检测用于获取当前仍被按着的键。

主要分支如下，按源码处理顺序整理：

| 输入情况 | 代码行为 |
|---|---|
| `pygame.QUIT` | `running=False` |
| 已结束后按R | 调用`reset()` |
| 已结束后按ESC | `running=False` |
| 未结束时按方向键或WASD | 调用`snake.set_direction(...)` |
| 未结束时按P | `paused = not paused` |
| 未结束时按R | 调用`reset()` |
| 未结束时按ESC | `running=False` |

连续按键只在没有结束、没有暂停时处理：

```python
# 持续按键检测（长按方向键也能响应）
if not self.game_over and not self.paused:
    keys = pygame.key.get_pressed()
    if keys[pygame.K_UP] or keys[pygame.K_w]:
        self.snake.set_direction(Direction.UP)
    elif keys[pygame.K_DOWN] or keys[pygame.K_s]:
        self.snake.set_direction(Direction.DOWN)
    elif keys[pygame.K_LEFT] or keys[pygame.K_a]:
        self.snake.set_direction(Direction.LEFT)
    elif keys[pygame.K_RIGHT] or keys[pygame.K_d]:
        self.snake.set_direction(Direction.RIGHT)
    self.speed_boost = keys[pygame.K_SPACE]
```

这一段位于`handle_events()`内部。`elif`使同一轮持续按键检测最多采用一个方向；空格状态另行记录。

`reset()`重新创建蛇与食物，恢复初始分数和速度：

```python
def reset(self):
    """重置游戏状态"""
    self.snake = Snake()
    self.food = Food()
    self.score = 0
    self.high_score = self._load_high_score()
    self.running = True
    self.paused = False
    self.game_over = False
    self.move_timer = 0
    self.move_interval = 150  # 毫秒，越小越快
    self.speed_boost = False   # 空格加速
    self.total_time = 0
    self.food.spawn(self.snake.body, pygame.time.get_ticks())
```

暂停没有退出主循环：窗口仍处理输入和重绘，只是`update()`不推进游戏逻辑。暂停时事件分支仍可能缓存方向，恢复后采用的方向也要以实际操作为准。


## 11. 移动计时、吃食物与速度提升

渲染频率与蛇的移动频率分开控制，画面可以持续刷新，蛇按指定间隔前进一格。

`Game.update()`原代码：

```python
def update(self, dt):
    """更新游戏逻辑"""
    if self.game_over or self.paused:
        return

    self.total_time += dt
    # 按住空格键时，时间累积加倍（速度翻倍）
    boost_multiplier = 2 if self.speed_boost else 1
    self.move_timer += dt * boost_multiplier

    current_time = pygame.time.get_ticks()
    self.food.update_animation(current_time)

    if self.move_timer >= self.move_interval:
        self.move_timer = 0
        alive = self.snake.move()

        if not alive:
            self.game_over = True
            if self.score > self.high_score:
                self.high_score = self.score
                self._save_high_score()
            return

        # 吃到食物
        if self.snake.head == self.food.position:
            self.snake.grow()
            self.score += 10
            self.food.spawn(self.snake.body, current_time)
            # 每吃5个食物加速
            if self.score % 50 == 0:
                self.move_interval = max(50, self.move_interval - 10)
```

逻辑顺序不能调换理解：**暂停/结束检查 → 累计时间 → 达到间隔后移动 → 碰撞结束 → 检查食物 → 加分、重生成食物与提速**。

| 条件 | 参数变化 |
|---|---|
| 新一局 | `move_interval=150`毫秒 |
| 吃一个食物 | `score += 10`，设置增长标记 |
| 分数达到50的倍数 | 间隔缩短10毫秒 |
| 间隔已到50毫秒 | 不再缩短 |
| 按住空格 | 累计时间乘2，不修改基础间隔 |

基础速度等级用`(150 - move_interval) // 10 + 1`显示。150毫秒为1级，140毫秒为2级；空格加速单独显示。

源码每轮最多移动一次，移动后把计时器清零。实际速度受到帧间隔影响，所以“加速x2”描述的是累计倍率，不能据此断言任何负载下都精确两倍。


## 12. 绘制窗口、蛇、食物与状态提示

`Game.__init__()`用Pygame创建窗口、时钟和字体，再调用`reset()`。主要初始化代码：

```python
pygame.init()
self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
pygame.display.set_caption("🐍 贪吃蛇")
self.clock = pygame.time.Clock()
self.font_large = pygame.font.SysFont("Microsoft YaHei", 36, bold=True)
self.font_medium = pygame.font.SysFont("Microsoft YaHei", 20)
self.font_small = pygame.font.SysFont("Microsoft YaHei", 14)
self.font_mono = pygame.font.SysFont("Consolas", 16)

self.reset()
```

以上片段位于实例方法内，`self`指当前Game对象。`Surface`是Pygame中的绘图表面，窗口和半透明遮罩都可以用它承载像素。

绘制顺序由`draw()`确定：

```python
def draw(self):
    """绘制整个画面"""
    self.screen.fill(COLOR_BG)
    self.draw_grid()
    self.food.draw(self.screen, pygame.time.get_ticks())
    self.snake.draw(self.screen)
    self.draw_ui()

    if self.game_over:
        self.draw_game_over()

    pygame.display.flip()
```

| 绘制部分 | 当前实现 |
|---|---|
| 背景 | 深色底色、20像素间距的网格线 |
| 蛇 | 圆角矩形，头部带眼睛，身体交替绿色 |
| 食物 | 红色圆形、高光和半透明光晕 |
| 普通UI | 分数、最高分、基础速度等级、加速提示 |
| 暂停 | 中央显示“暂停中” |
| 结束 | 半透明黑色遮罩、得分、重开和退出提示 |

`pygame.display.flip()`把这一轮完成的画面更新到窗口。碰撞检测仍使用逻辑坐标，圆角、眼睛和光晕只是外观。

源码依赖系统字体`Microsoft YaHei`。如果系统没有该字体，尤其在macOS上，需要检查中文是否显示为方块；窗口能打开不代表字体显示正确。


## 13. 最高分的读取与保存

最高分需要跨进程保留，源码使用一个文本文件，不需要数据库。

`Game`中的读写方法：

```python
def _load_high_score(self):
    """从文件读取最高分"""
    try:
        with open("high_score.txt", "r") as f:
            return int(f.read().strip())
    except (FileNotFoundError, ValueError):
        return 0

def _save_high_score(self):
    """保存最高分到文件"""
    try:
        with open("high_score.txt", "w") as f:
            f.write(str(self.high_score))
    except OSError:
        pass
```

读取时将文本转换为整数；文件不存在或内容不是整数时返回0。写入时用`"w"`覆盖原内容。`with`保证退出代码块时关闭文件。

注意“保存文件”与“更新最高分变量”是两步：

| 时机 | 当前源码行为 |
|---|---|
| 新一局`reset()` | 从文件加载最高分 |
| 撞自身且本局得分更高 | 更新`high_score`并保存 |
| 正常退出主循环 | 保存当前`high_score` |
| 尚未撞自身就按R | 重置，未先比较本局得分 |
| 尚未撞自身就退出 | 保存旧`high_score`，未先把本局得分更新进去 |

因此，当前版本只有结束碰撞分支会比较本局得分与最高分；不能直接把它描述为“退出一定保存本局新纪录”。另外，写入失败被`except OSError: pass`忽略，界面不会提示保存失败。

结束界面的“新纪录”条件为`score >= high_score and score > 0`，平纪录也可能显示这个提示。这里记录源码行为，没有修改原项目。


## 14. 根据源码核对游戏行为

运行后应先核对基础交互，再检查规则与状态。下表是依据源码整理的操作清单，**本次没有实际运行游戏，结果均需在本地验证**。

| 检查项 | 操作 | 当前源码对应的结果 |
|---|---|---|
| 启动 | 在项目目录运行入口 | 600×400窗口，蛇长3，自动向右走 |
| 改向 | 方向键或WASD | 下一次移动采用合法方向 |
| 禁止反向 | 向右移动时按左 | 保持向右 |
| 穿墙 | 让蛇越过任一边界 | 从对应另一侧进入 |
| 吃食物 | 头部进入食物逻辑坐标 | 加10分，下一次成功移动增长 |
| 食物位置 | 多次吃到食物 | 新逻辑坐标不在蛇身上 |
| 自身碰撞 | 让头部进入不会离开的身体格子 | 停止移动，出现结束层 |
| 暂停/继续 | 连续两次按P | 停止逻辑更新，再恢复 |
| 重开 | 运行、暂停或结束时按R | 分数0、长度3、间隔150毫秒，继续运行 |
| 基础提速 | 达到50、100等分数 | 间隔分别降到140、130等毫秒 |
| 空格加速 | 按住再松开空格 | 累计倍率2，再恢复1 |
| 最高分 | 超过记录后撞自身，再重新运行 | 从文件读回记录 |
| 退出 | ESC或关闭窗口 | 主循环结束并释放窗口 |

需要特别观察两个源码边界：

1. **食物动画与逻辑位置可能逐渐错位。**`Food.update_animation()`使用的公式是：

```python
self.bob_offset = 0.2 * (current_time / 200.0) ** 0.5
```

绘制纵坐标再加`int(self.bob_offset * 4)`。这个量随进程经过时间增长，不是上下往复的周期动画；逻辑食物坐标没有同时变化。长时间运行时，应核对画面中的食物与可吃到的格子是否一致。

2. **满棋盘没有独立胜利分支。**`Food.spawn()`无空位时保持原位置；它不会自动宣布胜利。

反馈问题时，把“源码实际怎样做”与“希望改成什么规则”分开。例如：

```text
当前源码在越过边界时穿墙，但文件顶部说明写着撞墙结束。
请先确认我想要的规则，再统一说明与实现；不要同时大改其他功能。
```

如果出现真实异常，提供运行命令、Python解释器、完整报错和复现步骤。


## 15. 运行中容易遇到的问题

| 现象 | 当前案例优先检查 |
|---|---|
| `No module named pygame` | 安装依赖和运行脚本是否使用同一个`.venv`解释器 |
| Pygame安装失败 | 错误中是否显示Python版本或平台缺少可用构建 |
| 窗口闪退 | 从终端启动，保留异常堆栈；不要只双击文件 |
| 按键无响应 | 游戏窗口是否获得焦点，是否已经结束或暂停 |
| 中文显示为方块 | 系统是否提供源码指定的中文字体 |
| 最高分文件不在项目目录 | 当前工作目录是否正确 |
| 本局新高没有保存 | 是否在撞自身前直接退出或重开，参照上一节的保存时机 |
| 看到食物但吃不到 | 核对动画像素偏移与逻辑坐标 |

依赖问题可以先执行Windows检查命令：

```powershell
Get-Location
.\.venv\Scripts\python.exe -c "import sys; print(sys.executable)"
.\.venv\Scripts\python.exe -c "import pygame; print(pygame.version.ver)"
```

macOS用`./.venv/bin/python`替换解释器路径即可。GUI程序还需要可用的桌面显示环境，纯终端服务器上不能把“导入成功”当成“图形界面验证完成”。


## 16. 来源与核验范围

- 课程：[VibeCoding实战百度网盘](https://pan.baidu.com/s/1zQT5QFTIAOSm-e5cnSYy5A?pwd=1234)，提取码1234。
- 对应视频：`视频/07-入门案例-贪吃蛇GUI游戏开发.mp4`。
- 主要依据：网盘在线代码预览中的`项目/snake/snake_game.py`完整420行，以及`项目/snake/requirements.txt`。
- 配套知识库：[VibeCoding零基础实战开发](https://www.yuque.com/xxcls/vibecoding)。其中[入门实战项目](https://www.yuque.com/xxcls/vibecoding/wmmswml3wi874b7v)讲的是记账APP，未将其流程当作贪吃蛇课的原文。
- uv操作参考：[Using Python environments](https://docs.astral.sh/uv/pip/environments/)。这是适应本仓库环境习惯的补充命令，不是原视频命令。
- 整理日期：2026-10-09。
- 代码片段保留源码的变量、参数和逻辑，网页代码行中的不换行空格还原为普通空格。没有复制项目配置或改变原源码。
- 本次完成资料读取与笔记整理；未安装Pygame、未运行游戏，未将复现提示词认定为课程原提示词。
